#### Import packages

In [158]:
import os
import numpy as np
import time
import csv 
import subprocess

import motor_utils
import flipper_utils

#### Motor settings

In [220]:
def set_motor_angle(ang_1:float, ang_2:float):
    #connect to motor
    motor_utils.init('COM6')
    motor_utils.setup()

    #set angles
    motor_utils.move(position=ang_1, MOTOR_ID=1, speed=20)
    time.sleep(1)
    motor_utils.move(position=ang_2, MOTOR_ID=2, speed=20)
    time.sleep(1)

    #close connection to motor
    motor_utils.deinit()

def eta_to_angle(desired_eta, m1ang_zero, m2ang_zero, move_motors:bool=False):
    theta =np.arccos(np.sqrt(desired_eta))/2
    print("theta (degrees)= ", np.degrees(theta))
    hwp_ang1 = m1ang_zero + np.degrees(theta)   # for matrix w
    hwp_ang2 = m2ang_zero - np.degrees(theta)   # for matrix inversed w
    if move_motors:
        set_motor_angle(hwp_ang1, hwp_ang2)
    return hwp_ang1, hwp_ang2


#### Flipper settings

In [200]:

def flipper_state(flipper_ID, block:bool):
    position = flipper_ID.get_position()
    if block == True and position == 2:
        flipper_ID.flip(1)
    elif block == False and position == 1:
        flipper_ID.flip(2)

flipper1 = flipper_utils.Controller(
    'COM8',                 # pick the correct COM port
    name='MFF101',          # optional change name
    verbose=True,           # optional verbose
    very_verbose=False      # optional very_verbose
    )

#flipper2 = flipper_utils.Controller(
    #'COM8',                 # pick the correct COM port
    #name='MFF101',          # optional change name
    #verbose=True,           # optional verbose
    #very_verbose=False      # optional very_verbose
    #)

MFF101: opening... done.
MFF101: getting info
MFF101: -> model number  = MFF002  
MFF101: -> type          = 16
MFF101: -> serial number = 37011596
MFF101: -> firmware version = 65539
MFF101: -> hardware version = 2
MFF101: getting position...
MFF101: -> position = 2


#### Remote functions:


In [ ]:
#Quick fetch of Displacement from SERF321 computer

def fetch_remote_dis():
    result = subprocess.run([
        "ssh",
        "serf321@10.46.40.133",
        "python --version &&",
        r'python C:\Users\SERF321\Documents\YuGroupCode\Time_Tagger\Swabian\displacement_calib_nogate.py'
    ],
        stdout = subprocess.PIPE,
        stderr = subprocess.PIPE,
        #capture_output=True,
        text=True
    )

    alpha_sum = float(result.stdout.splitlines()[-1])
    #print("Alpha Sum:", alpha_sum)
    print(result.stderr)
    return alpha_sum

#Quick fetch of SPDC from SERF321 computer

def fetch_remote_SPDC():
    result = subprocess.run([
        "ssh",
        "serf321@10.46.40.133",
        "python --version &&",
        r'python C:\Users\SERF321\Documents\YuGroupCode\Time_Tagger\Swabian\SPDC_check.py'
    ],
        stdout = subprocess.PIPE,
        stderr = subprocess.PIPE,
        #capture_output=True,
        text=True
    )

    check = result.stdout.splitlines()
    #print("Alpha Sum:", alpha_sum)
    print(result.stderr)
    print(check)

#### Dark measurement

Flip down the flipper if it is not already down

In [ ]:
#set flippers
flipper_state(flipper1, block=True)
#flipper_state(flipper2, block=True)

#Take measurements remotely with SSH
d_dk = fetch_remote_dis()
print("Dark displacement from remote measurement:", d_dk)

MFF101: getting position...
MFF101: -> position = 1
Flipper 1 is in down position -- blocked!

Dark displacement from remote measurement: 0.030886


#### Block displacment, set both HWP's
Calibrate matrix eta to HWP angle  

In [221]:
#set flippers
flipper_state(flipper1, block=False)
#flipper_state(flipper2, block=True)

#adjust HWP to zero-angle which is eta = 1
m1ang_zero = 8.5
m2ang_zero = 0.98
eta_to_angle(1, m1ang_zero, m2ang_zero, move_motors=True)



MFF101: getting position...
MFF101: -> position = 2
theta (degrees)=  0.0
Opening COM6...
Port open: True
Initialization commands sent.
Moving motor  1  to  8.5
Moving motor  2  to  0.98
Stop response: d064A00000060BD00000019FFEB0200001E01
Serial port closed: True


(np.float64(8.5), np.float64(0.98))

#### Check SPDC heralding
measure effective single photon efficiency

In [ ]:
fetch_remote_SPDC()

#### Block SPDC, inject Displacment
measure photon number distribution to calibrate mu

In [ ]:
#set flippers
flipper_state(flipper1, block=True)
#flipper_state(flipper2, block=False)

#Take measurements remotely with SSH
d_set = fetch_remote_dis()
print("Displacement value from remote measurement:", d_set)

MFF101: getting position...
MFF101: -> position = 1
Flipper 1 is in down position -- blocked!
MFF101: closing... done.



0.051449999999999996

#### Open both flippers
measure joint two-port distribution

In [ ]:
#set flippers
flipper_state(flipper1, block=False)
#flipper_state(flipper2, block=False)